# Common Mind Survey - Exploratory Data Analysis

This notebook performs EDA on the Typeform survey export data.

**Setup:** Run all cells in order from top to bottom.

In [ ]:
# Install required packages (run once)
!pip install -q pandas numpy matplotlib seaborn plotly openpyxl openai scikit-learn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', None)

# Plot settings
sns.set_style('whitegrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

## 1. Load Raw Data

In [ ]:
# Load the raw survey data
DATA_PATH = '/content/drive/MyDrive/_Client_Files/CommonMind/ai-driven-search-report/'
FILENAME = 'responses-cBLhVOUI-01KG5YNCSM9KM8SYBNTTQ98AC7-XPX692KPUA2130X3ZAN5TKT4.csv'

df_raw = pd.read_csv(DATA_PATH + FILENAME)
print(f"Loaded {len(df_raw)} responses, {len(df_raw.columns)} columns")

## 2. Data Cleaning - Explicit Multi-Select Mapping

Typeform exports multi-select questions as separate columns (one per option).
This section consolidates them back into single columns with lists.

In [ ]:
# Get column names list
cols = df_raw.columns.tolist()

# Define multi-select groups based on Typeform question mapping
# Format: 'NewColumnName': (start_col_1indexed, end_col_1indexed)
multiselect_groups = {
    'Q2_Industry': (3, 4),
    'Q9_AI_Goals': (11, 16),
    'Q13_Tracking_Blockers': (20, 25),
    'Q14_Content_Types_Published': (26, 34),
    'Q17_Page_Elements': (37, 41),
    'Q18_Review_Platforms': (42, 52),
    'Q19_Tactics_Current': (53, 63),
    'Q20_Tactics_Planned': (64, 74),
    'Q21_Measurement_Methods': (75, 81),
    'Q23_Bottlenecks': (83, 91),
    'Q24_Content_To_Prioritize': (92, 102),
    'Q25_Tactics_2026': (103, 110),
}

# Single-answer columns to rename
single_answer_cols = {
    'Q3_Company_Size': 5,
    'Q4_Changed_Approach': 6,
    'Q8_Strategy_Maturity': 10,
    'Q10_Traffic_YoY': 17,
    'Q11_GA4_LLM_Tracking': 18,
    'Q12_AI_Traffic_Percent': 19,
    'Q15_Publish_Pricing': 35,
    'Q16_Has_Schema': 36,
    'Q22_Content_Refresh_Frequency': 82,
}

print("Multi-select groups defined:")
for name, (start, end) in multiselect_groups.items():
    col_names = cols[start-1:end]
    print(f"  {name}: {len(col_names)} options")

In [ ]:
def consolidate_multiselect(df, groups):
    """Combine multi-select option columns into single list columns."""
    df_clean = df.copy()
    cols = df.columns.tolist()
    cols_to_drop = []

    for group_name, (start_1idx, end_1idx) in groups.items():
        start_idx = start_1idx - 1
        end_idx = end_1idx
        group_cols = cols[start_idx:end_idx]
        cols_to_drop.extend(group_cols)

        def get_selections(row, group_cols=group_cols):
            selected = []
            for col in group_cols:
                val = row[col]
                if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
                    selected.append(col)
            return selected if selected else None

        df_clean[group_name] = df_clean.apply(get_selections, axis=1)

    df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')
    return df_clean

# Apply consolidation
df_consolidated = consolidate_multiselect(df_raw, multiselect_groups)
print(f"Columns reduced: {len(df_raw.columns)} -> {len(df_consolidated.columns)}")

In [ ]:
# Rename columns for clarity
rename_map = {}

# Rename single-answer columns
for new_name, col_1idx in single_answer_cols.items():
    old_name = df_raw.columns[col_1idx - 1]
    if old_name in df_consolidated.columns:
        rename_map[old_name] = new_name

# Rename other key columns
additional_renames = {
    '#': 'Response_ID',
    'What is your primary email address for us to contact you?': 'Email',
    '*How important is it* for your brand to appear in _AI search/LLMs_?': 'Q5_Importance_AI_Search',
    '*How important is it* for your brand to maintain visibility in _traditional search engines_ (Google, Bing, etc)?': 'Q6_Importance_Traditional_Search',
    '*How confident do you feel* that your brand will maintain visibility as AI search & LLMs evolve?': 'Q7_Confidence_Visibility',
    '*How confident are you that your team can publish enough content to maintain or grow visibility* across traditional search (Google) and/or AI search platforms (ChatGPT, Perplexity, etc) in 2026?': 'Q26_Confidence_Content_2026',
    'What is your *biggest fear about AI-driven search for your brand?*': 'Q27_Biggest_Fear',
    "What's the #1 experiment or tactic you've tried (or want to try) to improve AI/AEO visibility? \n\nWhat, if any, results are you seeing from it? ": 'Q28_Tactics_Tried',
    'What question do you hope this report will answer for leaders like you?': 'Q29_Question_For_Report',
}
rename_map.update(additional_renames)

df_clean = df_consolidated.rename(columns=rename_map)

# Create working copy
df = df_clean.copy()

print("\nFinal columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3}. {col[:70]}")

In [ ]:
# Verify: Check a sample row
print("=== SAMPLE ROW (first response) ===")
row = df.iloc[0]

for col in list(multiselect_groups.keys())[:4]:  # Show first 4 multi-selects
    if col in df.columns:
        print(f"\n{col}:")
        val = row[col]
        if val:
            for item in val:
                print(f"    - {item}")
        else:
            print("    (none selected)")

## 3. Missing Data Analysis

In [ ]:
print("=== MISSING DATA ANALYSIS ===\n")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_report = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_report = missing_report[missing_report['Missing'] > 0].sort_values('Percent', ascending=False)
print(missing_report.head(20))

print("\nNote: For multi-select columns, None means 'none selected' - valid data, not missing.")

## 4. Executive Summary Stats

In [ ]:
print("=" * 60)
print("EXECUTIVE SUMMARY - AI SEARCH READINESS")
print("=" * 60)

n = len(df)
print(f"\nTotal Responses: {n}")

# Key metrics using new column names
if 'Q5_Importance_AI_Search' in df.columns:
    ai_importance = df['Q5_Importance_AI_Search']
    print(f"\nAI Search Importance (1-5 scale):")
    print(f"  Average: {ai_importance.mean():.2f}")
    print(f"  % Rating 4-5 (High): {(ai_importance >= 4).mean()*100:.1f}%")

if 'Q7_Confidence_Visibility' in df.columns:
    confidence = df['Q7_Confidence_Visibility']
    print(f"\nConfidence in AI Visibility (1-5 scale):")
    print(f"  Average: {confidence.mean():.2f}")
    print(f"  % Rating 4-5 (Confident): {(confidence >= 4).mean()*100:.1f}%")

# The importance-confidence gap
if 'Q5_Importance_AI_Search' in df.columns and 'Q7_Confidence_Visibility' in df.columns:
    gap = df['Q5_Importance_AI_Search'].mean() - df['Q7_Confidence_Visibility'].mean()
    print(f"\nINSIGHT: Importance-Confidence Gap: {gap:.2f}")
    print(f"   (Brands know AI search matters but aren't confident they're ready)")

## 5. EDA Visualizations

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Company Size Distribution
ax1 = axes[0, 0]
if 'Q3_Company_Size' in df.columns:
    size_order = ['1-10', '11-50', '51-200', '201-1,000', '1,000-5,000', '5,001+']
    size_counts = df['Q3_Company_Size'].value_counts()
    size_counts = size_counts.reindex([s for s in size_order if s in size_counts.index])
    size_counts.plot(kind='bar', ax=ax1, color=colors[1])
    ax1.set_title('Company Size Distribution', fontsize=12, fontweight='bold')
    ax1.set_xlabel('')
    ax1.tick_params(axis='x', rotation=45)

# 2. AI Importance Distribution
ax2 = axes[0, 1]
if 'Q5_Importance_AI_Search' in df.columns:
    df['Q5_Importance_AI_Search'].value_counts().sort_index().plot(kind='bar', ax=ax2, color=colors[2])
    ax2.set_title('AI Search Importance Ratings', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Rating (1-5)')

# 3. AI Importance vs Traditional Search Importance
ax3 = axes[1, 0]
if 'Q5_Importance_AI_Search' in df.columns and 'Q6_Importance_Traditional_Search' in df.columns:
    ax3.scatter(df['Q6_Importance_Traditional_Search'], df['Q5_Importance_AI_Search'], alpha=0.5, c=colors[2], s=100)
    ax3.plot([1, 5], [1, 5], 'k--', alpha=0.3, label='Equal importance')
    ax3.set_xlabel('Traditional Search Importance')
    ax3.set_ylabel('AI Search Importance')
    ax3.set_title('AI vs Traditional Search Importance', fontsize=12, fontweight='bold')
    ax3.legend()

# 4. Organic Traffic Change
ax4 = axes[1, 1]
if 'Q10_Traffic_YoY' in df.columns:
    traffic_counts = df['Q10_Traffic_YoY'].value_counts()
    traffic_counts.index = traffic_counts.index.str.replace('*', '', regex=False)
    traffic_counts.plot(kind='bar', ax=ax4, color=colors[3])
    ax4.set_title('YoY Organic Traffic Change', fontsize=12, fontweight='bold')
    ax4.set_xlabel('')
    ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Strategy Maturity
if 'Q8_Strategy_Maturity' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    maturity_counts = df['Q8_Strategy_Maturity'].value_counts()
    maturity_counts.plot(kind='barh', ax=ax, color='#3498db')
    ax.set_title('AI Strategy Maturity Levels', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Respondents')
    
    total = len(df['Q8_Strategy_Maturity'].dropna())
    for i, (idx, val) in enumerate(maturity_counts.items()):
        ax.text(val + 0.5, i, f'{val/total*100:.0f}%', va='center')
    
    plt.tight_layout()
    plt.show()

## 6. Cross-Tabulation Analysis

In [ ]:
# Confidence by Company Size
print("=== CONFIDENCE BY COMPANY SIZE ===\n")
if 'Q7_Confidence_Visibility' in df.columns and 'Q3_Company_Size' in df.columns:
    avg_conf = df.groupby('Q3_Company_Size')['Q7_Confidence_Visibility'].mean().sort_values(ascending=False)
    print("Average Confidence by Company Size:")
    print(avg_conf.round(2))

## 7. Clustering Analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Prepare data for clustering - use numeric columns
cluster_cols = [
    'Q5_Importance_AI_Search',
    'Q6_Importance_Traditional_Search',
    'Q7_Confidence_Visibility',
    'Q26_Confidence_Content_2026'
]

cluster_cols = [c for c in cluster_cols if c in df.columns]
print(f"Clustering on {len(cluster_cols)} features: {cluster_cols}")

df_cluster = df[cluster_cols].dropna()
print(f"Rows with complete data: {len(df_cluster)}")

# Scale and cluster
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_cluster['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize with PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df_cluster['Cluster'], cmap='viridis', alpha=0.6, s=100)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Respondent Segments (PCA Visualization)', fontsize=14, fontweight='bold')
plt.show()

print(f"\nCluster sizes:")
print(df_cluster['Cluster'].value_counts().sort_index())

In [ ]:
# Profile each cluster
print("\n" + "="*60)
print("CLUSTER PROFILES")
print("="*60)

cluster_profiles = df_cluster.groupby('Cluster')[cluster_cols].mean().round(2)
print(cluster_profiles.T)

print("\nSEGMENT INTERPRETATION:")
for cluster_id in range(n_clusters):
    profile = cluster_profiles.loc[cluster_id]
    avg_importance = profile.iloc[0]
    avg_confidence = profile.iloc[2] if len(profile) > 2 else profile.iloc[1]

    if avg_importance >= 4 and avg_confidence >= 4:
        name = "AI Leaders"
        desc = "High importance, high confidence - ahead of the curve"
    elif avg_importance >= 4 and avg_confidence < 3:
        name = "Anxious Aspirers"
        desc = "Know AI matters but not confident they're ready - PRIME PROSPECTS"
    elif avg_importance < 3 and avg_confidence >= 3:
        name = "Traditionalists"
        desc = "Not prioritizing AI search yet"
    else:
        name = "Wait-and-See"
        desc = "Low urgency, low confidence - need education"

    print(f"\n  Cluster {cluster_id}: {name}")
    print(f"    {desc}")
    print(f"    AI Importance: {avg_importance:.1f}, Confidence: {avg_confidence:.1f}")

## 8. AI-Powered Text Analysis

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

# Get the key
api_key = getpass('Enter OpenAI API key: ')

# Create client with explicit key
client = OpenAI(api_key=api_key)

# Test it
try:
    test = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Say 'working'"}],
        max_tokens=10
    )
    print("API key working:", test.choices[0].message.content)
except Exception as e:
    print("Error:", e)

In [ ]:
def analyze_with_ai(responses, question_context):
    """Use GPT-4o to analyze open-ended survey responses."""
    responses_text = "\n---\n".join([str(r) for r in responses if pd.notna(r)][:50])

    prompt = f"""You are analyzing survey responses from B2B marketing leaders about AI-driven search.

Question asked: {question_context}

Analyze these {len(responses)} responses and provide:

1. **Top 5 Themes** - Most common concerns/approaches mentioned (with % estimate)
2. **Key Quotes** - 3-4 verbatim quotes that best represent the sentiment
3. **Actionable Insights** - What should a marketing agency tell these leaders?
4. **Segment Differences** - Any patterns by sophistication level?

Responses:
{responses_text}

Format your response with clear headers and bullet points."""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

# Analyze fears
if 'Q27_Biggest_Fear' in df.columns:
    print("="*60)
    print("AI ANALYSIS: BIGGEST FEARS")
    print("="*60)
    fears = df['Q27_Biggest_Fear'].dropna().tolist()
    fear_analysis = analyze_with_ai(fears, "What is your biggest fear about AI-driven search for your brand?")
    print(fear_analysis)

In [ ]:
# Analyze tactics
if 'Q28_Tactics_Tried' in df.columns:
    print("\n" + "="*60)
    print("AI ANALYSIS: TACTICS & EXPERIMENTS")
    print("="*60)
    tactics = df['Q28_Tactics_Tried'].dropna().tolist()
    tactic_analysis = analyze_with_ai(tactics, "What's the #1 experiment or tactic you've tried to improve AI/AEO visibility?")
    print(tactic_analysis)

In [ ]:
# Generate executive summary
def generate_executive_summary(df):
    stats = f"""
    Survey: AI-Driven Search Readiness ({len(df)} B2B marketing leaders)

    Key Metrics:
    - Average AI Search Importance: {df['Q5_Importance_AI_Search'].mean():.2f}/5
    - Average Confidence: {df['Q7_Confidence_Visibility'].mean():.2f}/5
    - Company sizes: {df['Q3_Company_Size'].value_counts().head(3).to_dict()}

    Traffic trends:
    {df['Q10_Traffic_YoY'].value_counts().to_dict()}
    """

    prompt = f"""You are a senior marketing strategist writing an executive summary for a report on "The State of AI-Driven Search in B2B Marketing."

Based on this survey data, write a compelling 3-paragraph executive summary that:
1. Opens with the key tension/finding (importance vs. confidence gap)
2. Highlights 2-3 surprising or actionable insights
3. Ends with a clear call-to-action for marketing leaders

Data:
{stats}

Write in a professional but engaging tone. Use specific numbers. Keep it under 250 words."""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4
    )
    return response.choices[0].message.content

print("\n" + "="*60)
print("AI-GENERATED EXECUTIVE SUMMARY")
print("="*60)
exec_summary = generate_executive_summary(df)
print(exec_summary)

## 9. Save Cleaned Data

In [ ]:
# Save cleaned dataframe for future use
output_path = DATA_PATH + 'survey_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned data saved to: {output_path}")